# Тренировка Модели U-Net

## Импортируем зависимости

In [1]:
%pip install qai_hub
%pip install qai_hub_models
%pip install torch

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 774.8 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 3.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 6.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 2.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.3/85.3 kB 776.8 kB/s eta 0:00:00 0:00:01
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 760.1 kB/s eta 0:00:00 0:00:01
Using cached tqdm-4.67.3-py3-non

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from qai_hub_models.models.unet_segmentation import Model
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

## Скачиваем модель

In [4]:
torch_model = Model.from_pretrained()

Downloading: "https://github.com/milesial/Pytorch-UNet/zipball/master" to /home/oogis/.cache/torch/hub/master.zip


In [5]:
input_shape = torch_model.get_input_spec()
sample_inputs = torch_model.sample_inputs()

In [6]:
print(input_shape)

{'image': ((1, 3, 640, 1280), 'float32')}


In [7]:
print(torch_model)

UNet(
  (model): UNet(
    (inc): DoubleConv(
      (double_conv): Sequential(
        (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
      )
    )
    (down1): Down(
      (maxpool_conv): Sequential(
        (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (1): DoubleConv(
          (double_conv): Sequential(
            (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
            (3): Conv2d(128, 128, kernel_si

In [10]:
class WatermarkRemovalUNet(nn.Module):
    def __init__(self, pretrained=True, freeze_encoder=True):
        super(WatermarkRemovalUNet, self).__init__()
        
        print("Загрузка предобученной модели...")
        self.base_unet = Model.from_pretrained() if pretrained else Model()
        
        # Сохраняем энкодер и декодер до последнего слоя
        self.encoder = nn.ModuleList([
            self.base_unet.model.inc,
            self.base_unet.model.down1,
            self.base_unet.model.down2,
            self.base_unet.model.down3,
            self.base_unet.model.down4
        ])
        
        self.decoder = nn.ModuleList([
            self.base_unet.model.up1,
            self.base_unet.model.up2,
            self.base_unet.model.up3,
            self.base_unet.model.up4
        ])
        
        # Замораживаем энкодер если нужно
        if freeze_encoder:
            for module in self.encoder:
                for param in module.parameters():
                    param.requires_grad = False
        
        # Новый выходной слой для RGB (вместо outc с 2 каналами)
        self.output_layer = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 3, kernel_size=1),
            nn.Tanh()  # или Sigmoid, зависит от нормализации
        )
        
        # Опционально: добавляем остаточное соединение
        self.use_residual = True
    
    def forward(self, x):
        # Сохраняем для skip connections
        skip_connections = []
        
        # Encoder
        for i, encoder_layer in enumerate(self.encoder):
            x = encoder_layer(x)
            if i < len(self.encoder) - 1:  # Все кроме последнего
                skip_connections.append(x)
        
        # Decoder
        for i, decoder_layer in enumerate(self.decoder):
            x = decoder_layer(x, skip_connections[-(i+1)])
        
        # Выходной слой
        output = self.output_layer(x)
        
        # Остаточное соединение (помогает сохранить структуру)
        if self.use_residual:
            # Восстанавливаем оригинальный размер если нужно
            if output.shape[-2:] != skip_connections[0].shape[-2:]:
                output = F.interpolate(output, size=skip_connections[0].shape[-2:], mode='bilinear')
            
            # Добавляем остаточную связь с первым skip connection
            output = output + skip_connections[0][:, :3, :, :]  # Берем только RGB каналы
        
        return torch.sigmoid(output)  # Нормализуем в [0, 1]
        

    def preprocess_image(self, image):
        """
        Предобработка изображения:
        - Ресайз до 224x224
        - Конвертация в тензор
        - Нормализация
        """
        if isinstance(image, str):
            # Если путь к файлу
            image = Image.open(image).convert('RGB')
        elif isinstance(image, np.ndarray):
            # Если numpy array (OpenCV)
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        
        # Применяем трансформации
        image_tensor = self.transform(image)
        
        # Добавляем batch dimension если нужно
        if len(image_tensor.shape) == 3:
            image_tensor = image_tensor.unsqueeze(0)
        
        return image_tensor
    
    def postprocess_image(self, tensor):
        """
        Постобработка тензора обратно в изображение
        """
        # Убираем batch dimension если есть
        if len(tensor.shape) == 4:
            tensor = tensor.squeeze(0)
        
        # Конвертируем в numpy
        image_np = tensor.detach().cpu().numpy()
        
        # Если в формате [C, H, W], меняем на [H, W, C]
        if image_np.shape[0] == 3:
            image_np = np.transpose(image_np, (1, 2, 0))
        
        # Ограничиваем значения и конвертируем в uint8
        image_np = np.clip(image_np * 255, 0, 255).astype(np.uint8)
        
        return image_np
    

# Тестируем модель

    # Создаем модель
model = WatermarkRemovalUNet(pretrained=True, freeze_encoder=True)
    
    # Проверяем архитектуру
print("\n" + "="*50)
print("Архитектура модели:")
print("="*50)
print(model)
    
    # Тестовый forward pass
batch_size = 1
test_input = torch.randn(batch_size, 3, 224, 224)  # Размер как в модели
print(f"\nТестовый вход: {test_input.shape}")
    
with torch.no_grad():
        output = model(test_input)
        print(f"Тестовый выход: {output.shape}")
        print(f"Диапазон значений: [{output.min():.4f}, {output.max():.4f}]")
    
    # Проверяем обучаемые параметры
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
print(f"\nВсего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")

Загрузка предобученной модели...


Using cache found in /home/oogis/.cache/torch/hub/milesial_Pytorch-UNet_master



Архитектура модели:
WatermarkRemovalUNet(
  (base_unet): UNet(
    (model): UNet(
      (inc): DoubleConv(
        (double_conv): Sequential(
          (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (5): ReLU(inplace=True)
        )
      )
      (down1): Down(
        (maxpool_conv): Sequential(
          (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
          (1): DoubleConv(
            (double_conv): Sequential(
              (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_runni

In [8]:
import os
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

class WatermarkDataset(Dataset):
    """Датасет для обучения удаления вотермарок"""
    def __init__(self, root_dir):
        self.root_dir = root_dir
        
        # Предполагаем структуру:
        # root_dir/
        #   ├── watermarked/
        #   └── clean/
        
        self.watermarked_dir = os.path.join(root_dir, "watermark")
        self.clean_dir = os.path.join(root_dir, "no-watermark")
        
        # Получаем список файлов
        self.watermarked_files = sorted([f for f in os.listdir(self.watermarked_dir) 
                                         if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
        self.clean_files = sorted([f for f in os.listdir(self.clean_dir) 
                                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))])
        
        # Проверяем соответствие файлов
        if len(self.watermarked_files) != len(self.clean_files):
            print(f"ВНИМАНИЕ: Количество файлов не совпадает!")
            print(f"Watermarked: {len(self.watermarked_files)}, Clean: {len(self.clean_files)}")
            
            # Берем минимальное количество
            min_len = min(len(self.watermarked_files), len(self.clean_files))
            self.watermarked_files = self.watermarked_files[:min_len]
            self.clean_files = self.clean_files[:min_len]
    
    def __len__(self):
        return len(self.watermarked_files)
    
    def __getitem__(self, idx):
        try:
            # Загружаем изображения
            wm_path = os.path.join(self.watermarked_dir, self.watermarked_files[idx])
            clean_path = os.path.join(self.clean_dir, self.clean_files[idx])
            
            # Читаем как RGB
            wm_img = Image.open(wm_path).convert('RGB')
            clean_img = Image.open(clean_path).convert('RGB')
            
            # Ресайз до 224x224 (размер модели)
            wm_img = wm_img.resize((224, 224))
            clean_img = clean_img.resize((224, 224))
            
            # Конвертируем в тензоры
            wm_tensor = torch.from_numpy(np.array(wm_img)).float() / 255.0
            clean_tensor = torch.from_numpy(np.array(clean_img)).float() / 255.0
            
            # Permute: [H, W, C] -> [C, H, W]
            wm_tensor = wm_tensor.permute(2, 0, 1)
            clean_tensor = clean_tensor.permute(2, 0, 1)
            
            return wm_tensor, clean_tensor
            
        except Exception as e:
            print(f"Ошибка при загрузке изображения {idx}: {e}")
            # Возвращаем нулевые тензоры в случае ошибки
            return torch.zeros((3, 224, 224)), torch.zeros((3, 224, 224))

def train_model(model, dataloader, epochs=1, device='cuda'):
    """Обучение модели"""
    
    # Проверяем доступность CUDA
    if device == 'cuda' and not torch.cuda.is_available():
        print("CUDA недоступна, переключаюсь на CPU")
        device = 'cpu'
    
    model = model.to(device)
    
    # Оптимизатор
    optimizer = optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=0.001
    )
    
    criterion = nn.L1Loss()
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
    
    print(f"Начало обучения на {device}")
    print(f"Всего батчей: {len(dataloader)}")
    print("-" * 50)
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        batch_count = 0
        
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        try:
            # Простая итерация без enumerate
            for batch_idx, (wm_images, clean_images) in enumerate(dataloader):
                batch_count += 1
                
                # Проверка формы данных
                if batch_idx == 0:
                    print(f"Размер батча: wm_images={wm_images.shape}, clean_images={clean_images.shape}")
                
                # Перенос данных
                wm_images = wm_images.to(device)
                clean_images = clean_images.to(device)
                
                # Forward
                optimizer.zero_grad()
                outputs = model(wm_images)
                loss = criterion(outputs, clean_images)
                
                # Backward
                loss.backward()
                optimizer.step()
                
                running_loss += loss.item()
                
                # Простой вывод каждые 10 батчей
                if batch_idx % 10 == 0:
                    avg_loss_current = running_loss / (batch_idx + 1)
                    print(f'  Batch {batch_idx}/{len(dataloader)} | '
                          f'Loss: {loss.item():.4f} | Avg: {avg_loss_current:.4f}')
            
            # Завершение эпохи
            avg_loss = running_loss / len(dataloader)
            scheduler.step()
            
            print(f"\nEpoch {epoch+1}/{epochs} завершён. Avg Loss: {avg_loss:.4f}")
            
            # Сохранение
            if (epoch + 1) % 2 == 0:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': avg_loss,
                }, f'checkpoint_epoch_{epoch+1}.pth')
                print(f"Чекпоинт сохранен: checkpoint_epoch_{epoch+1}.pth")
                
        except Exception as e:
            print(f"Ошибка во время обучения эпохи {epoch+1}: {e}")
            import traceback
            traceback.print_exc()
            break
    
    return model

# Основной код

    # Проверяем датасет
print("Создание датасета...")
dataset = WatermarkDataset("wm-nowm/train")
print(f"Размер датасета: {len(dataset)}")
    
    # Проверяем первый элемент
if len(dataset) > 0:
        wm, clean = dataset[0]
        print(f"Пример данных: wm shape={wm.shape}, clean shape={clean.shape}")
        print(f"Диапазон значений: wm [{wm.min():.3f}, {wm.max():.3f}], "
              f"clean [{clean.min():.3f}, {clean.max():.3f}]")
    
    # Создаем DataLoader БЕЗ num_workers для отладки
dataloader = DataLoader(
        dataset, 
        batch_size=4, 
        shuffle=True, 
        num_workers=0,  # ИСПОЛЬЗУЕМ 0 ДЛЯ ОТЛАДКИ
        pin_memory=False
    )
    
    # Проверяем модель
print("\nПроверка модели...")
    # Здесь должен быть ваш код создания модели
    
    # Обучение
print("\nНачало обучения...")
    # model = ваша_модель
    # trained_model = train_model(model, dataloader, epochs=25)

Создание датасета...
ВНИМАНИЕ: Количество файлов не совпадает!
Watermarked: 12510, Clean: 12477
Размер датасета: 12477
Пример данных: wm shape=torch.Size([3, 224, 224]), clean shape=torch.Size([3, 224, 224])
Диапазон значений: wm [0.000, 1.000], clean [0.000, 1.000]

Проверка модели...

Начало обучения...


In [ ]:
print("Начало обучения...")
print('cuda' if torch.cuda.is_available() else 'cpu')
trained_model = train_model(model, dataloader, epochs=25, device='cuda' if torch.cuda.is_available() else 'cpu')
    
    # 4. Сохраняем финальную модель
torch.save(trained_model.state_dict(), "watermark_removal_model.pth")
print("Модель сохранена!")
    

Начало обучения...
cpu
Начало обучения на cpu
Всего батчей: 3120
--------------------------------------------------

Epoch 1/25
Размер батча: wm_images=torch.Size([4, 3, 224, 224]), clean_images=torch.Size([4, 3, 224, 224])
  Batch 0/3120 | Loss: 0.2396 | Avg: 0.2396
  Batch 10/3120 | Loss: 0.3055 | Avg: 0.2616
  Batch 20/3120 | Loss: 0.2899 | Avg: 0.2640
  Batch 30/3120 | Loss: 0.2243 | Avg: 0.2596
  Batch 40/3120 | Loss: 0.2294 | Avg: 0.2568
  Batch 50/3120 | Loss: 0.2902 | Avg: 0.2553
  Batch 60/3120 | Loss: 0.2983 | Avg: 0.2534


c:\Users\molik\anaconda3\Lib\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  Batch 70/3120 | Loss: 0.2198 | Avg: 0.2549
  Batch 80/3120 | Loss: 0.2675 | Avg: 0.2548
  Batch 90/3120 | Loss: 0.3076 | Avg: 0.2542
  Batch 100/3120 | Loss: 0.2411 | Avg: 0.2532
  Batch 110/3120 | Loss: 0.2455 | Avg: 0.2522
  Batch 120/3120 | Loss: 0.2882 | Avg: 0.2523
  Batch 130/3120 | Loss: 0.2233 | Avg: 0.2519
  Batch 140/3120 | Loss: 0.2449 | Avg: 0.2519
  Batch 150/3120 | Loss: 0.2839 | Avg: 0.2520
  Batch 160/3120 | Loss: 0.2808 | Avg: 0.2516
  Batch 170/3120 | Loss: 0.3511 | Avg: 0.2521
  Batch 180/3120 | Loss: 0.2614 | Avg: 0.2516
  Batch 190/3120 | Loss: 0.2392 | Avg: 0.2516
  Batch 200/3120 | Loss: 0.2361 | Avg: 0.2522
  Batch 210/3120 | Loss: 0.2422 | Avg: 0.2525
  Batch 220/3120 | Loss: 0.2699 | Avg: 0.2526
  Batch 230/3120 | Loss: 0.1904 | Avg: 0.2523
  Batch 240/3120 | Loss: 0.2377 | Avg: 0.2526
  Batch 250/3120 | Loss: 0.2254 | Avg: 0.2531
  Batch 260/3120 | Loss: 0.2465 | Avg: 0.2531
  Batch 270/3120 | Loss: 0.2421 | Avg: 0.2537
  Batch 280/3120 | Loss: 0.2663 | Avg